In [9]:
# ============================================================
# SETUP
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import statsmodels.api as sm
import json
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/siap'

freq = pd.read_csv(f'{BASE}/data/freq_raw.csv')
sev  = pd.read_csv(f'{BASE}/data/sev_raw.csv')

print(f"freq : {freq.shape} | sev : {sev.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
freq : (678013, 12) | sev : (26639, 2)


In [10]:
# ============================================================
# BLOC 1 — NETTOYAGE DES ANOMALIES (cf. script 2)
# ============================================================
freq['ClaimNb'] = freq['ClaimNb'].clip(upper=4)
freq = freq[freq['Exposure'] > 0.01].copy()

p99 = sev['ClaimAmount'].quantile(0.99)
sev['ClaimAmount_raw'] = sev['ClaimAmount'].copy()
sev['ClaimAmount'] = sev['ClaimAmount'].clip(upper=p99)

print(f"Polices après filtre exposition : {len(freq)}")
print(f"Seuil winsorisation (p99) : {p99:.0f} €")

Polices après filtre exposition : 664410
Seuil winsorisation (p99) : 16794 €


In [11]:
# ============================================================
# BLOC 2 — INTERPRÉTATION ACTUARIELLE DES COEFFICIENTS
# ============================================================
# En Poisson log-link, exp(coef) = "rate ratio"
# C'est LA quantité que les actuaires présentent en commission tarifs

rate_ratios = np.exp(glm_freq.params).sort_values(ascending=False)

print("=== Top 10 facteurs qui AUGMENTENT le risque ===")
print(rate_ratios.head(10))

print("\n=== Top 10 facteurs qui DIMINUENT le risque ===")
print(rate_ratios.tail(10))

# Exemple de lecture :
# Si "BonusMalus_band_5_very_bad" a un rate ratio de 1.8
# => ces conducteurs ont une fréquence 80% plus élevée
# que la catégorie de référence (1_excellent, droppée par drop_first)

# On vérifie aussi les p-values : un coefficient avec p > 0.05
# n'est statistiquement pas différent de 0 → candidat à retirer
non_significatif = glm_freq.pvalues[glm_freq.pvalues > 0.05]
print(f"\nVariables non significatives (p > 0.05) : {len(non_significatif)}")
print(non_significatif.index.tolist())

=== Top 10 facteurs qui AUGMENTENT le risque ===
BonusMalus_band_4_bad        1.472726
Region_R74                   1.280881
BonusMalus_band_2_good       1.251519
Region_R21                   1.222688
Region_R94                   1.192373
young_and_bad                1.146335
Area_F                       1.131068
VehBrand_B12                 1.128892
Region_R53                   1.125342
BonusMalus_band_3_neutral    1.120676
dtype: float64

=== Top 10 facteurs qui DIMINUENT le risque ===
Region_R73                    0.892168
DrivAge_band_3_experienced    0.863575
VehBrand_B14                  0.862117
Region_R41                    0.792037
DrivAge_band_4_mature         0.770945
Region_R83                    0.761835
DrivAge_band_2_adult          0.724466
VehAge_band_1_recent          0.689396
DrivAge_band_5_senior         0.644614
const                         0.022280
dtype: float64

Variables non significatives (p > 0.05) : 29
['young_and_bad', 'Area_B', 'Area_C', 'Area_D', 'Area_E'

In [12]:
# ============================================================
# BLOC 3 — REGROUPEMENT DES MODALITÉS RARES
# ============================================================
# Une modalité représentant moins de 2% du portefeuille est
# fusionnée dans "Other" — pratique standard pour stabiliser
# les estimations et éviter le bruit dans les p-values

seuil = 0.02 * len(freq)

region_counts = freq['Region'].value_counts()
brand_counts  = freq['VehBrand'].value_counts()

rare_regions = region_counts[region_counts < seuil].index.tolist()
rare_brands  = brand_counts[brand_counts < seuil].index.tolist()

print(f"Régions regroupées en 'Other' ({len(rare_regions)}) : {rare_regions}")
print(f"Marques regroupées en 'Other' ({len(rare_brands)})  : {rare_brands}")

freq['Region']   = freq['Region'].replace(rare_regions, 'Other')
freq['VehBrand'] = freq['VehBrand'].replace(rare_brands, 'Other')

print(f"\nModalités Region après regroupement   : {freq['Region'].nunique()}")
print(f"Modalités VehBrand après regroupement : {freq['VehBrand'].nunique()}")

Régions regroupées en 'Other' (11) : ['R41', 'R25', 'R26', 'R23', 'R22', 'R83', 'R74', 'R94', 'R21', 'R42', 'R43']
Marques regroupées en 'Other' (3)  : ['B11', 'B13', 'B14']

Modalités Region après regroupement   : 12
Modalités VehBrand après regroupement : 9


In [14]:
# ============================================================
# BLOC 4 — ENCODAGE ET SPLIT TRAIN/TEST (CORRIGÉ)
# ============================================================

from sklearn.model_selection import train_test_split

# ✔ FIX : création des variables *_band (manquantes)
freq['DrivAge_band'] = pd.cut(freq['DrivAge'], bins=[18, 30, 45, 60, 80, 100], labels=False)
freq['BonusMalus_band'] = pd.cut(freq['BonusMalus'], bins=[50, 80, 100, 120, 150, 300], labels=False)
freq['VehAge_band'] = pd.cut(freq['VehAge'], bins=[0, 1, 3, 5, 10, 20, 50], labels=False)

cat_cols = ['Area', 'VehBrand', 'VehGas', 'Region',
            'DrivAge_band', 'BonusMalus_band', 'VehAge_band']

# sécurité (évite KeyError si colonne absente)
cat_cols = [c for c in cat_cols if c in freq.columns]

freq_encoded = pd.get_dummies(freq, columns=cat_cols, drop_first=True, dtype=int)

freq_train, freq_test = train_test_split(
    freq_encoded,
    test_size=0.2,
    random_state=42,
    stratify=(freq_encoded['ClaimNb'] > 0).astype(int)
)

exclude = ['IDpol', 'ClaimNb', 'Exposure']
X_cols = [c for c in freq_train.columns if c not in exclude]

print(f"Colonnes après encodage : {freq_encoded.shape[1]}")
print(f"Nombre de features (X_cols) : {len(X_cols)}")
print(f"Train : {len(freq_train)} | Test : {len(freq_test)}")

Colonnes après encodage : 46
Nombre de features (X_cols) : 43
Train : 531528 | Test : 132882


In [15]:
# ============================================================
# BLOC 5 — GLM POISSON POUR LA FRÉQUENCE
# ============================================================
X_train = freq_train[X_cols].astype(float)
y_train = freq_train['ClaimNb']
exposure_train = freq_train['Exposure']

X_test = freq_test[X_cols].astype(float)
y_test = freq_test['ClaimNb']
exposure_test = freq_test['Exposure']

X_train_sm = sm.add_constant(X_train)
X_test_sm  = sm.add_constant(X_test)

glm_freq = sm.GLM(
    y_train,
    X_train_sm,
    family=sm.families.Poisson(),
    offset=np.log(exposure_train)
).fit()

print(glm_freq.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               531528
Model:                            GLM   Df Residuals:                   531484
Model Family:                 Poisson   Df Model:                           43
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.1187e+05
Date:                Fri, 12 Jun 2026   Deviance:                   1.6885e+05
Time:                        15:48:36   Pearson chi2:                 9.57e+05
No. Iterations:                     7   Pseudo R-squ. (CS):            0.01245
Covariance Type:            nonrobust                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -4.0485    

In [16]:
# ============================================================
# BLOC 6 — INTERPRÉTATION ET VARIABLES NON SIGNIFICATIVES
# ============================================================
rate_ratios = np.exp(glm_freq.params).sort_values(ascending=False)

print("=== Top 10 facteurs qui AUGMENTENT le risque ===")
print(rate_ratios.head(10))

print("\n=== Top 10 facteurs qui DIMINUENT le risque ===")
print(rate_ratios.tail(10))

non_significatif = glm_freq.pvalues[glm_freq.pvalues > 0.05]
print(f"\nVariables non significatives (p > 0.05) : {len(non_significatif)}")
print(non_significatif.index.tolist())

=== Top 10 facteurs qui AUGMENTENT le risque ===
VehAge_band_5.0     1.996681
Area_F              1.399257
VehAge_band_4.0     1.318185
Area_E              1.258556
DrivAge_band_2.0    1.231170
Area_D              1.177974
Region_R53          1.145072
VehBrand_B12        1.142715
Region_R82          1.118626
Region_R24          1.109085
dtype: float64

=== Top 10 facteurs qui DIMINUENT le risque ===
BonusMalus_band_2.0    0.939641
VehAge                 0.928869
Region_R72             0.924158
Region_R73             0.906869
BonusMalus_band_1.0    0.813594
VehAge_band_2.0        0.811214
BonusMalus_band_3.0    0.801833
VehAge_band_1.0        0.655730
BonusMalus_band_4.0    0.282015
const                  0.017448
dtype: float64

Variables non significatives (p > 0.05) : 16
['Density', 'VehBrand_B10', 'VehBrand_B2', 'VehBrand_B3', 'VehBrand_B4', 'VehBrand_B6', 'VehBrand_Other', 'Region_R11', 'Region_R31', 'Region_R52', 'Region_R54', 'Region_R91', 'Region_R93', 'DrivAge_band_3.0', 'DrivA

In [17]:
# ============================================================
# BLOC 7 — ÉVALUATION FRÉQUENCE
# ============================================================
y_pred_train = glm_freq.predict(X_train_sm, offset=np.log(exposure_train))
y_pred_test  = glm_freq.predict(X_test_sm,  offset=np.log(exposure_test))

def poisson_deviance(y_true, y_pred):
    eps = 1e-9
    y_pred = np.clip(y_pred, eps, None)
    terme = np.where(y_true == 0, 0, y_true * np.log(y_true / y_pred))
    return 2 * np.mean(terme - (y_true - y_pred))

dev_train = poisson_deviance(y_train.values, y_pred_train.values)
dev_test  = poisson_deviance(y_test.values, y_pred_test.values)
print(f"Déviance Poisson train : {dev_train:.4f}")
print(f"Déviance Poisson test  : {dev_test:.4f}")

freq_obs_test  = y_test.sum() / exposure_test.sum()
freq_pred_test = y_pred_test.sum() / exposure_test.sum()
print(f"\nFréquence observée (test)  : {freq_obs_test:.4f}")
print(f"Fréquence prédite (test)   : {freq_pred_test:.4f}")

Déviance Poisson train : 0.3177
Déviance Poisson test  : 0.3199

Fréquence observée (test)  : 0.0998
Fréquence prédite (test)   : 0.0995


In [18]:
# ============================================================
# BLOC 8 — GLM GAMMA POUR LA SÉVÉRITÉ
# ============================================================
freq_full = pd.concat([freq_train, freq_test], axis=0)

sev_full = sev.merge(
    freq_full[['IDpol'] + X_cols],
    on='IDpol',
    how='inner'
)
print(f"Sinistres avec caractéristiques police : {sev_full.shape}")

sev_train, sev_test = train_test_split(sev_full, test_size=0.2, random_state=42)

X_sev_train = sm.add_constant(sev_train[X_cols].astype(float))
y_sev_train = sev_train['ClaimAmount']

X_sev_test = sm.add_constant(sev_test[X_cols].astype(float))
y_sev_test = sev_test['ClaimAmount']

glm_sev = sm.GLM(
    y_sev_train,
    X_sev_train,
    family=sm.families.Gamma(link=sm.families.links.Log())
).fit()

print(glm_sev.summary())

Sinistres avec caractéristiques police : (26340, 46)
                 Generalized Linear Model Regression Results                  
Dep. Variable:            ClaimAmount   No. Observations:                21072
Model:                            GLM   Df Residuals:                    21028
Model Family:                   Gamma   Df Model:                           43
Link Function:                    Log   Scale:                          2.0259
Method:                          IRLS   Log-Likelihood:            -1.7940e+05
Date:                Fri, 12 Jun 2026   Deviance:                       21924.
Time:                        15:49:20   Pearson chi2:                 4.26e+04
No. Iterations:                    20   Pseudo R-squ. (CS):           0.006340
Covariance Type:            nonrobust                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------

In [19]:
# ============================================================
# BLOC 9 — ÉVALUATION SÉVÉRITÉ
# ============================================================
y_sev_pred_test = glm_sev.predict(X_sev_test)

mae_sev = np.mean(np.abs(y_sev_test - y_sev_pred_test))
cout_moyen_obs  = y_sev_test.mean()
cout_moyen_pred = y_sev_pred_test.mean()

print(f"MAE sévérité (test)        : {mae_sev:.2f} €")
print(f"Coût moyen observé (test)  : {cout_moyen_obs:.2f} €")
print(f"Coût moyen prédit (test)   : {cout_moyen_pred:.2f} €")

MAE sévérité (test)        : 1066.26 €
Coût moyen observé (test)  : 1565.75 €
Coût moyen prédit (test)   : 1562.31 €


In [20]:
# ============================================================
# BLOC 10 — SAUVEGARDE
# ============================================================
os.makedirs(f'{BASE}/models', exist_ok=True)

# 10a. Objets pickle (pour analyses futures dans Colab)
with open(f'{BASE}/models/glm_freq.pkl', 'wb') as f:
    pickle.dump(glm_freq, f)
with open(f'{BASE}/models/glm_sev.pkl', 'wb') as f:
    pickle.dump(glm_sev, f)

# 10b. Colonnes utilisées (avec const) — ordre exact requis pour la prédiction
with open(f'{BASE}/models/glm_columns.json', 'w') as f:
    json.dump(X_train_sm.columns.tolist(), f)

# 10c. Export portable des coefficients (pour le backend, sans statsmodels)
export = {
    "glm_freq": {
        "coefficients": glm_freq.params.to_dict(),
        "link": "log",
        "offset": "log(Exposure)",
        "family": "poisson"
    },
    "glm_sev": {
        "coefficients": glm_sev.params.to_dict(),
        "link": "log",
        "family": "gamma"
    },
    "feature_columns": X_train_sm.columns.tolist()
}
with open(f'{BASE}/models/glm_coefficients.json', 'w') as f:
    json.dump(export, f, indent=2)

# 10d. On sauvegarde aussi les nouvelles données encodées et propres
# (utiles pour le script 4 — XGBoost/LightGBM travaillera sur les mêmes splits)
freq_train.to_csv(f'{BASE}/data/freq_train_v2.csv', index=False)
freq_test.to_csv(f'{BASE}/data/freq_test_v2.csv',  index=False)
sev_full.to_csv(f'{BASE}/data/sev_full_v2.csv', index=False)

with open(f'{BASE}/data/feature_cols_v2.json', 'w') as f:
    json.dump(X_cols, f)

print("GLM Poisson et GLM Gamma sauvegardés (v2 — nettoyé).")

GLM Poisson et GLM Gamma sauvegardés (v2 — nettoyé).
